In [19]:
import numpy as np
import pandas as pd
import warnings
import numpy as np

from itertools import combinations, product
from math import comb
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef
)

from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

## File

In [20]:
df = pd.read_csv(r"../../../data/processed/data_vif.csv")
target = "Risk_Label"

n_total = len(df)
n_train = int(n_total * 0.5)
n_valid = int(n_total * 0.3)
n_test = n_total - n_train - n_valid

#Date 컬럼 제거
df.drop("Date", axis=1, inplace=True)

# Risk_Label 매핑 (High Risk=1, Low Risk=0)
df["Risk_Label"] = df["Risk_Label"].map({"Low Risk": 0, "High Risk": 1})

#train/valid/test 분할 5/3/2
train_df = df.iloc[:n_train].reset_index(drop=True)
valid_df = df.iloc[n_train : n_train + n_valid].reset_index(drop=True)
test_df = df.iloc[n_train + n_valid :].reset_index(drop=True)

print(f"total rows: {n_total}, train: {len(train_df)}, valid: {len(valid_df)}, test: {len(test_df)}")

# Split features/target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_valid = valid_df.drop(columns=[target])
y_valid = valid_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

print("train shape:", X_train.shape, y_train.shape)
print("valid shape:", X_valid.shape, y_valid.shape)
print("test shape:", X_test.shape, y_test.shape)

total rows: 4108, train: 2054, valid: 1232, test: 822
train shape: (2054, 28) (2054,)
valid shape: (1232, 28) (1232,)
test shape: (822, 28) (822,)


In [21]:
scaler = MinMaxScaler().set_output(transform="pandas")

# train에만 fit하고 valid/test에는 같은 scaler를 적용
X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)
X_test = scaler.transform(X_test)

print('train/valid/test:', len(X_train), len(X_valid), len(X_test))
print('y_train class:', y_train.value_counts().to_dict())
print('y_valid class:', y_valid.value_counts().to_dict())
print('y_test class:', y_test.value_counts().to_dict())

train/valid/test: 2054 1232 822
y_train class: {0: 1859, 1: 195}
y_valid class: {0: 1070, 1: 162}
y_test class: {0: 730, 1: 92}


In [ ]:
# =========================
# Param Search Space
# =========================

param_space = {
    "C": [0.1,0.5, 1.0, 3.0, 5.0, 10.0],  # L2 규제 강도 (작을수록 강한 규제)
    "class_weight": ["balanced"],  
    "solver": ["liblinear"],             
    "threshold": [0.1, 0.2, 0.3, 0.35, 0.40, 0.45, 0.50, 0.55]
}


def make_param_list(param_space):
    keys = list(param_space.keys())
    values = list(param_space.values())

    param_list = []
    for comb in product(*values):
        param_list.append(dict(zip(keys, comb)))

    return param_list


param_list = make_param_list(param_space)

총 파라미터 조합 수: 48


[{'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.1},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.2},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.3},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.35},
 {'C': 0.1,
  'class_weight': 'balanced',
  'solver': 'liblinear',
  'threshold': 0.4}]

## 전역선택법

In [17]:
def get_metrics(y_true, y_proba, threshold=0.5):
    y_pred = (y_proba >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    gmean = np.sqrt(recall * spec)

    metrics = {
        "gmean": gmean,
        "acc": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision,
        "recall": recall,
        "spec": spec,
        "balanced_acc": balanced_accuracy_score(y_true, y_pred),
        "mcc": matthews_corrcoef(y_true, y_pred),
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "threshold": threshold,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp
    }

    return metrics


def forward_selection_lr(
    X_train,
    y_train,
    X_valid,
    y_valid,
    param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
):
    """
    Logistic Regression 기반 전진선택법.
    기준 지표는 validation gmean.
    더 이상 gmean이 min_delta 이상 개선되지 않으면 자동 종료.
    """

    remaining_features = list(X_train.columns)
    selected_features = []

    best_score = -np.inf
    best_model = None
    best_params = None
    best_metrics = None

    step_history = []
    candidate_history = []

    step = 1

    while len(remaining_features) > 0:
        step_best = None

        if verbose:
            print(f"\n[STEP {step}]")
            print(f"현재 선택 변수: {selected_features if selected_features else '없음'}")
            print(f"후보 변수 수: {len(remaining_features)}")

        for feature in remaining_features:
            trial_features = selected_features + [feature]

            for params in param_list:
                threshold = params["threshold"]

                model_params = {
                    "C": params["C"],
                    "class_weight": params["class_weight"],
                    "solver": params["solver"]
                }

                model = LogisticRegression(
                    penalty="l2",
                    max_iter=max_iter,
                    random_state=random_state,
                    **model_params
                )

                model.fit(X_train[trial_features], y_train)

                y_proba = model.predict_proba(X_valid[trial_features])[:, 1]

                metrics = get_metrics(
                    y_true=y_valid,
                    y_proba=y_proba,
                    threshold=threshold
                )

                record = {
                    "step": step,
                    "added_feature": feature,
                    "features": trial_features,
                    "params": params,
                    "metrics": metrics,
                    "model": model
                }

                candidate_history.append({
                    "step": step,
                    "added_feature": feature,
                    "n_features": len(trial_features),
                    **params,
                    **metrics
                })

                if step_best is None:
                    step_best = record
                else:
                    if metrics["gmean"] > step_best["metrics"]["gmean"]:
                        step_best = record

        improvement = step_best["metrics"]["gmean"] - best_score

        if improvement > min_delta:
            selected_features = step_best["features"]
            remaining_features.remove(step_best["added_feature"])

            best_score = step_best["metrics"]["gmean"]
            best_model = step_best["model"]
            best_params = step_best["params"]
            best_metrics = step_best["metrics"]

            step_history.append({
                "step": step,
                "added_feature": step_best["added_feature"],
                "n_features": len(selected_features),
                "improvement": improvement,
                **best_params,
                **best_metrics
            })

            if verbose:
                print(
                    f"선택 변수: {step_best['added_feature']} | "
                    f"gmean={best_score:.4f} | "
                    f"threshold={best_params['threshold']} | "
                    f"C={best_params['C']} | "
                    f"class_weight={best_params['class_weight']}"
                )

            step += 1

        else:
            if verbose:
                print(f"개선폭 {improvement:.6f} <= min_delta {min_delta}")
                print("전진선택 종료")

            break

    result = {
        "selected_features": selected_features,
        "best_model": best_model,
        "best_params": best_params,
        "best_metrics": best_metrics,
        "step_history": pd.DataFrame(step_history),
        "candidate_history": pd.DataFrame(candidate_history)
    }

    return result

In [18]:
forward_result = forward_selection_lr(
    X_train=X_train,
    y_train=y_train,
    X_valid=X_valid,
    y_valid=y_valid,
    param_list=param_list,
    min_delta=0.001,
    random_state=1,
    max_iter=2000,
    verbose=True
)


print("\n" + "=" * 80)
print("최종 선택 변수")
print("=" * 80)
print(forward_result["selected_features"])


print("\n" + "=" * 80)
print("Best Parameters")
print("=" * 80)
print(forward_result["best_params"])


print("\n" + "=" * 80)
print("Validation Metrics")
print("=" * 80)

metric_order = [
    "gmean",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "mcc",
    "roc_auc",
    "pr_auc",
    "tn",
    "fp",
    "fn",
    "tp"
]

metrics_df = pd.DataFrame([forward_result["best_metrics"]])[metric_order]
display(metrics_df)


print("\n" + "=" * 80)
print("Validation Confusion Matrix")
print("row=true, col=pred")
print("[[TN, FP],")
print(" [FN, TP]]")
print("=" * 80)

cm = np.array([
    [forward_result["best_metrics"]["tn"], forward_result["best_metrics"]["fp"]],
    [forward_result["best_metrics"]["fn"], forward_result["best_metrics"]["tp"]]
])

print(cm)


print("\n" + "=" * 80)
print("Step History")
print("=" * 80)

step_cols = [
    "step",
    "added_feature",
    "n_features",
    "gmean",
    "improvement",
    "threshold",
    "acc",
    "f1",
    "precision",
    "recall",
    "spec",
    "balanced_acc",
    "tn",
    "fp",
    "fn",
    "tp",
    "C",
    "class_weight",
    "solver"
]

display(forward_result["step_history"][step_cols])


# test set이 있으면 test 평가
try:
    best_model = forward_result["best_model"]
    selected_features = forward_result["selected_features"]
    best_threshold = forward_result["best_params"]["threshold"]

    y_test_proba = best_model.predict_proba(X_test[selected_features])[:, 1]

    test_metrics = get_metrics(
        y_true=y_test,
        y_proba=y_test_proba,
        threshold=best_threshold
    )

    print("\n" + "=" * 80)
    print("Test Metrics")
    print("=" * 80)

    test_metrics_df = pd.DataFrame([test_metrics])[metric_order]
    display(test_metrics_df)

    print("\n" + "=" * 80)
    print("Test Confusion Matrix")
    print("row=true, col=pred")
    print("[[TN, FP],")
    print(" [FN, TP]]")
    print("=" * 80)

    test_cm = np.array([
        [test_metrics["tn"], test_metrics["fp"]],
        [test_metrics["fn"], test_metrics["tp"]]
    ])

    print(test_cm)

except NameError:
    pass


[STEP 1]
현재 선택 변수: 없음
후보 변수 수: 28
선택 변수: NASDAQ_return(%) | gmean=0.6392 | threshold=0.5 | C=1.0 | class_weight=balanced

[STEP 2]
현재 선택 변수: ['NASDAQ_return(%)']
후보 변수 수: 27
선택 변수: KOSPI 200_BB_width | gmean=0.6595 | threshold=0.45 | C=5.0 | class_weight=balanced

[STEP 3]
현재 선택 변수: ['NASDAQ_return(%)', 'KOSPI 200_BB_width']
후보 변수 수: 26
선택 변수: KOSPI 200_PPO_Hist | gmean=0.6677 | threshold=0.45 | C=1.0 | class_weight=balanced

[STEP 4]
현재 선택 변수: ['NASDAQ_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist']
후보 변수 수: 25
선택 변수: KOSPI 200_OG | gmean=0.6771 | threshold=0.45 | C=10.0 | class_weight=balanced

[STEP 5]
현재 선택 변수: ['NASDAQ_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG']
후보 변수 수: 24
선택 변수: KODEX 200_Premium | gmean=0.6859 | threshold=0.45 | C=5.0 | class_weight=balanced

[STEP 6]
현재 선택 변수: ['NASDAQ_return(%)', 'KOSPI 200_BB_width', 'KOSPI 200_PPO_Hist', 'KOSPI 200_OG', 'KODEX 200_Premium']
후보 변수 수: 23
선택 변수: Signal2_Buy | gmean=0.6883 | threshold=0.45 |

,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.688329,0.45,0.660714,0.360856,0.239837,0.728395,0.650467,0.689431,0.261416,0.724311,0.348547,696,374,44,118



Validation Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[696 374]
 [ 44 118]]

Step History


,step,added_feature,n_features,gmean,improvement,threshold,acc,f1,precision,recall,spec,balanced_acc,tn,fp,fn,tp,C,class_weight,solver
0,1,NASDAQ_return(%),1,0.639233,inf,0.50,0.711851,0.336449,0.241287,0.555556,0.735514,0.645535,787,283,72,90,1.0,balanced,liblinear
1,2,KOSPI 200_BB_width,2,0.659533,0.020300,0.45,0.621753,0.332378,0.216418,0.716049,0.607477,0.661763,650,420,46,116,5.0,balanced,liblinear
2,3,KOSPI 200_PPO_Hist,3,0.667747,0.008215,0.45,0.627435,0.339568,0.221388,0.728395,0.612150,0.670272,655,415,44,118,1.0,balanced,liblinear
3,4,KOSPI 200_OG,4,0.677056,0.009309,0.45,0.650162,0.349925,0.231537,0.716049,0.640187,0.678118,685,385,46,116,10.0,balanced,liblinear
4,5,KODEX 200_Premium,5,0.685899,0.008842,0.45,0.660714,0.358896,0.238776,0.722222,0.651402,0.686812,697,373,45,117,5.0,balanced,liblinear
5,6,Signal2_Buy,6,0.688329,0.002431,0.45,0.660714,0.360856,0.239837,0.728395,0.650467,0.689431,696,374,44,118,5.0,balanced,liblinear



Test Metrics


,gmean,threshold,acc,f1,precision,recall,spec,balanced_acc,mcc,roc_auc,pr_auc,tn,fp,fn,tp
0,0.656484,0.45,0.607056,0.293217,0.183562,0.728261,0.591781,0.660021,0.203075,0.725968,0.274091,432,298,25,67



Test Confusion Matrix
row=true, col=pred
[[TN, FP],
 [FN, TP]]
[[432 298]
 [ 25  67]]


In [23]:
selected_features

['NASDAQ_return(%)',
 'KOSPI 200_BB_width',
 'KOSPI 200_PPO_Hist',
 'KOSPI 200_OG',
 'KODEX 200_Premium',
 'Signal2_Buy']